In [0]:
DESCRIBE proyecto.silver.propiedades;

col_name,data_type,comment
propiedad_id,bigint,Surrogate Key auto-generada para la propiedad
partido,string,Partido o municipio estandarizado
region,string,"Región geográfica: capital_federal, gba_norte, gba_oeste, gba_sur, etc."
tipo_operacion,string,Tipo de transacción estandarizada: alquiler o venta
precio_usd,"decimal(15,2)",Precio estandarizado en USD (ARS convertidos a tasa 1520)
expensas_usd,"decimal(15,2)",Expensas mensuales normalizadas a USD
ambientes,string,"Cantidad o categoría de ambientes: 1 a 9, 10-20, No Especificado"
m2_totales,"decimal(15,2)",Superficie total en metros cuadrados
m2_cubiertos,"decimal(15,2)",Superficie cubierta en metros cuadrados
antiguedad,int,Años de antigüedad (imputado con 999 cuando no está especificado)


In [0]:
-- Temporary View to Expand the EDA (CAST all STRING number values to FLOAT)

CREATE OR REPLACE TEMPORARY VIEW properties_clean_silver AS
SELECT 
    CASE WHEN precio RLIKE '^[0-9]+(\.[0-9]+)?$' THEN precio::double ELSE NULL END as precio,
    moneda,
    CASE WHEN ambientes RLIKE '^[0-9]+(\.[0-9]+)?$' THEN ambientes::double ELSE NULL END as ambientes,
    CASE WHEN metros_cuadrados_totales RLIKE '^[0-9]+(\.[0-9]+)?$' THEN metros_cuadrados_totales::double ELSE NULL END as m2_totales,
    CASE WHEN metros_cuadrados_cubiertos RLIKE '^[0-9]+(\.[0-9]+)?$' THEN metros_cuadrados_cubiertos::double ELSE NULL
    END as m2_cubiertos,
    CASE WHEN antiguedad RLIKE '^[0-9]+(\.[0-9]+)?$' THEN antiguedad::double ELSE NULL END as antiguedad,
    tipo_de_operacion,
    id,
    zona,
    ubicacion,
    expensas,
    piso,
    cochera,
    estado,
    url,
    fecha,
    source_file,
    ingestion_timestamp

FROM proyecto.bronze.properties_bronze;

In [0]:
SELECT COUNT(*) FROM properties_clean_silver;

COUNT(*)
1246717


In [0]:
CREATE OR REPLACE TEMPORARY VIEW properties_USD_silver AS (
  SELECT
    id, 
    tipo_de_operacion,
    CASE 
        WHEN moneda = 'ARS' THEN ROUND(precio / 1520, -1) ELSE precio END AS precio,
    ambientes,
    m2_totales,
    m2_cubiertos,
    antiguedad,
    zona,
    ubicacion,
    piso,
    cochera,
    estado,
    url,
    fecha,
    source_file,
    ingestion_timestamp
FROM properties_clean_silver
WHERE 
  precio IS NOT NULL
  AND precio > 0
  AND tipo_de_operacion IN ('venta', 'alquiler')
  AND moneda IN ('ARS','USD')
);

In [0]:
SELECT COUNT(*) FROM properties_USD_silver;

COUNT(*)
1221174


In [0]:
-- Vista nueva todo en USD y sin los outliers en el precio

CREATE OR REPLACE TEMPORARY VIEW properties_clean_silver_2 AS
(
WITH limites AS (
    SELECT 
      tipo_de_operacion,
      PERCENTILE(precio, 0.05) AS p05,
      PERCENTILE(precio, 0.95) AS p95
    FROM properties_USD_silver
    GROUP BY tipo_de_operacion
  )
  SELECT p.*
  FROM properties_usd_silver AS p
  JOIN limites AS l 
    ON p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p05 AND l.p95
);

In [0]:
SELECT COUNT(*) FROM properties_clean_silver_2;

COUNT(*)
1103035


In [0]:
SELECT 
    'BRONZE' as capa, 
    table_name, 
    table_type
FROM proyecto.information_schema.tables
WHERE table_schema = 'bronze'

UNION ALL

SELECT 
    'SILVER' as capa, 
    table_name, 
    table_type
FROM proyecto.information_schema.tables
WHERE table_schema = 'silver'

UNION ALL

SELECT 
    'GOLD' as capa, 
    table_name, 
    table_type
FROM proyecto.information_schema.tables
WHERE table_schema = 'gold'

ORDER BY capa, table_name;

capa,table_name,table_type
BRONZE,properties_bronze,MANAGED
GOLD,dim_caracteristicas,MANAGED
GOLD,dim_tiempo,MANAGED
GOLD,dim_tipo_operacion,MANAGED
GOLD,dim_zona,MANAGED
GOLD,fact_propiedades,MANAGED
SILVER,propiedades,MANAGED


In [0]:
SELECT 
    'Bronze (properties_bronze)' as tabla, 
    COUNT(*) as registros
FROM proyecto.bronze.properties_bronze

UNION ALL

SELECT 
    'Silver (propiedades)' as tabla, 
    COUNT(*) as registros
FROM proyecto.silver.propiedades;

tabla,registros
Bronze (properties_bronze),1246717
Silver (propiedades),956730
